### PREREQUISITES

In [1]:
import pandas as pd
import numpy as np
import datetime
import re
import warnings
# import datatable as dt

In [2]:
pd.options.mode.chained_assignment = None

In [26]:
arc_data_types = {'arc_code': int, 'arc_cat2': str}
master_data_types = {'esm_agerelax_codes': str}
# candidate_table_df = pd.read_csv(r"C:\Users\AshutoshMishra\OneDrive - Cubastion Consulting Pvt Ltd\Desktop\SSC OPS\RESULT PROCESSING MTS 2024\mts 2024\candidates.csv")
vacancy_table_df =  pd.read_csv(r"C:\Users\AshutoshMishra\OneDrive - Cubastion Consulting Pvt Ltd\Desktop\SSC OPS\RESULT PROCESSING MTS 2024\mts 2024\vacancy_table_for_result_processing.csv")
# master_table_df =  pd.read_csv(r"C:\Users\AshutoshMishra\OneDrive - Cubastion Consulting Pvt Ltd\Desktop\SSC OPS\RESULT PROCESSING MTS 2024\mts 2024\master_table.csv", dtype = master_data_types)
# cut_off_table_df = pd.read_csv(r"C:\Users\AshutoshMishra\OneDrive - Cubastion Consulting Pvt Ltd\Desktop\SSC OPS\RESULT PROCESSING MTS 2024\mts 2024\cut_off_table.csv")
# arc_table_df = pd.read_csv(r"C:\Users\AshutoshMishra\OneDrive - Cubastion Consulting Pvt Ltd\Desktop\SSC OPS\RESULT PROCESSING MTS 2024\mts 2024\arc_table.csv", dtype = arc_data_types)

In [4]:
origin_candidates_df = candidate_table_df.copy()

In [5]:
candidate_table_df.columns

Index(['id', 'regno', 'otrId', 'myApplicationId', 'emailId', 'phoneNo', 'name',
       'changed_name', 'father_name', 'mother_name',
       ...
       'pst_height_relaxation_code', 'pst_chest_not_expanded',
       'pst_chest_expanded', 'pst_status', 'pst_weight',
       'walk_male_1600m_15min', 'walk_female_1km_20min',
       'final_pet_pst_status', 'final_remarks', 'pt_finalstatus'],
      dtype='object', length=132)

In [6]:
# master_table_df

In [7]:
candidate_table_df['dob'] = pd.to_datetime(candidate_table_df['dob'])

In [8]:
candidate_table_df.shape

(2250749, 132)

In [9]:
candidate_table_df[['cutoff_flag', 'catsel_18_25', 'catsel_18_27', 'dob_flag_18_27', 'catsel_dob_18_27', 'dob_flag_18_25', 'catsel_dob_18_25', 'merit']] = None

In [10]:
candidate_table_df[['cutoff_flag', 'catsel_18_25', 'catsel_18_27', 'dob_flag_18_27', 'catsel_dob_18_27', 'dob_flag_18_25', 'catsel_dob_18_25', 'merit']].head(10)

,cutoff_flag,catsel_18_25,catsel_18_27,dob_flag_18_27,catsel_dob_18_27,dob_flag_18_25,catsel_dob_18_25,merit
0,None,None,None,None,None,None,None,None
1,None,None,None,None,None,None,None,None
2,None,None,None,None,None,None,None,None
3,None,None,None,None,None,None,None,None
4,None,None,None,None,None,None,None,None
5,None,None,None,None,None,None,None,None
6,None,None,None,None,None,None,None,None
7,None,None,None,None,None,None,None,None
8,None,None,None,None,None,None,None,None
9,None,None,None,None,None,None,None,None


In [11]:
candidate_table_df['debarred'].count()

1159

In [12]:
candidate_table_df.shape

(2250749, 140)

### MERIT

In [13]:
def get_merit(exam_name, serial_id, master_table, candidate_table_df):    
    master_record = master_table[(master_table['exam_name'] == exam_name) & (master_table['serial'] == serial_id)].iloc[0]
    candidates =  candidate_table_df[(candidate_table_df['rej_prov'].isnull()) & 
    (candidate_table_df['debarred'].isnull()) & 
    (candidate_table_df['total'].notnull())].sort_values(by = ['session2', 'part3_ga', 'session1', 'dob', 'cand_name'], ascending=[False, False, False, True, True])
    
    candidates['merit'] = range(1, len(candidates) + 1)

    rollno_to_merit = candidates.set_index('rollno')['merit'].to_dict()
    candidate_table_df['merit'] = candidate_table_df['rollno'].map(rollno_to_merit)
    
    return "{flag_name} updated successfully for {merit} candidates".format(
        flag_name=master_record['flag_name'],
        merit=len(candidates))

In [14]:
exam_name = 'MTS2024T1'
serial_id = 1
get_merit(exam_name, serial_id, master_table_df, candidate_table_df)

'merit updated successfully for 2249590 candidates'

In [15]:
candidate_table_df[candidate_table_df['merit'].isnull()].head(10)

,id,regno,otrId,myApplicationId,emailId,phoneNo,name,changed_name,father_name,mother_name,...,final_remarks,pt_finalstatus,cutoff_flag,catsel_18_25,catsel_18_27,dob_flag_18_27,catsel_dob_18_27,dob_flag_18_25,catsel_dob_18_25,merit
315481,bg317njuym8nfew,10005647986,a3ugqku9fd0wwe75,9ginqtzjh1x6p06,suman9608068275@gmail.com,9608068275,SUMAN KUMAR,NaN,MANIKANT SINGH,PUSHPA DEVI,...,NaN,NaN,None,None,None,None,None,None,None,NaN
315487,vrzvhsbzs7jhp8z,10008025646,9ki47mwjo847hjlt,x8285ljja6weuqg,ashukumar15082002@gmail.com,8076084878,ASHU KUMAR,NaN,NARESH KUMAR,JYOTI DEVI,...,NaN,NaN,None,None,None,None,None,None,None,NaN
315488,m8jjgliej1guu4u,10008080744,g90qz0v2zdlz2x44,0dcb84h8zr3wnho,kumarankit20148@gmail.com,6399055438,ANKIT KUMAR,NaN,SUSHIL KUMAR,SANGEETA,...,NaN,NaN,None,None,None,None,None,None,None,NaN
315489,pnih6lj44gqbb40,10005605293,wu6eeettt30jlupf,ao0egxcwl64ff4l,jakhardeepak306@gmail.com,9518059064,DEEPAK,NaN,SURESH,SUNITA,...,NaN,NaN,None,None,None,None,None,None,None,NaN
315491,p3lpom64hx7kyc4,10006535309,g1dug1zlsc91pa0h,i91dark5i9jw0jz,vishvendrasantha@gmail.com,9664465613,VISHVENDRA KUMAR MEENA,NaN,NITESH KUMAR MEENA,KAMLESH DEVI,...,NaN,NaN,None,None,None,None,None,None,None,NaN
315506,eenwujsegn9sc2x,10005675717,n4fcl7bn80tl5n5r,o0cy7kolf906d2z,rinkumeena850487@gmail.com,8504871526,RINKU KUMAR MEENA,NaN,SIYARAM MEENA,BHAVARO DEVI,...,NaN,NaN,None,None,None,None,None,None,None,NaN
315512,xecnt42pe4ii96d,10005782083,ezqbkatv2gx8gc5s,7z5wrw634plrzun,mohantamohanlal2@gmail.com,9348039949,MOHANLAL MOHANTA,NaN,RATNAKAR MOHANTA,RANJITA MOHANTA,...,NaN,NaN,None,None,None,None,None,None,None,NaN
315513,nqd8ac25ycqfskj,10001347835,h1cndbsts0bxmsgz,1cg6uxzo2g32urq,shivamtanwar0075@gmail.com,8813987623,SHIVAM,NaN,BHARPUR,BABITA DEVI,...,NaN,NaN,None,None,None,None,None,None,None,NaN
315514,ncgtdj4jdda739l,10006682774,uzs7q1ixtrmc9vt8,n7889epy0nk85fv,sonurajput774093@gmail.com,6378971704,SONU,NaN,MAHENDER SINGH,SAROJ DEVI,...,NaN,NaN,None,None,None,None,None,None,None,NaN
315515,lv19r39amx16rgp,10006134343,h61e2wup6p9n74su,xy8q037sok4f9vw,deepak35279@gmail.com,9708385909,DEEPAK RAJAK,NaN,RAMESH KUMAR RAJAK,VEENA DEVI,...,NaN,NaN,None,None,None,None,None,None,None,NaN


In [28]:
# columns_to_display = ["regno", "debarred"]  # Yeh aapke desired columns hain

# nan_rows = candidate_table_df[candidate_table_df["merit"].isna()][columns_to_display]
# nan_rows

In [18]:
candidate_table_df[candidate_table_df['merit'].notnull()][['merit']].head(2)

# candidate_table_df['merit'] = candidate_table_df['merit'].astype('Int64')

,merit
0,744253
1,1799946


In [21]:
candidate_table_df['cat3'].unique()
# candidate_table_df['cat3'] = candidate_table_df['cat3'].astype('Int64')
# candidate_table_df['cat3'] = candidate_table_df['cat3'].fillna(0)

<IntegerArray>
[0, 7, 4, 5, 8]
Length: 5, dtype: Int64

### CUTOFF FLAG

In [22]:
def get_cutoff(exam_name, serial_id, master_table_df, cut_off_table, candidates):
    master_record = master_table_df[(master_table_df['exam_name'] == exam_name) & (master_table_df['serial'] == serial_id)].iloc[0]
    candidates[master_record['flag_name']] = ''
    cutoff_dict = {}

    for index, row in cut_off_table.iterrows():
        category = int(row['category'])
        session1 = float(row['session1'])
        session2 = float(row['session2'])
        cutoff_dict[category] = [session1, session2]

    def calculate_cutoff(candidate):
        cutoff = ''

        if (candidate['session1'] >= cutoff_dict[9][0]) & (candidate['session2'] >= cutoff_dict[9][1]):
            cutoff += '9'

        for cat in [0, 1, 2, 6]:
            if (candidate['cat1'] == cat) & (cat in cutoff_dict):
                if (candidate['session1'] >= cutoff_dict[cat][0]) & (candidate['session2'] >= cutoff_dict[cat][1]):
                    cutoff += str(cat)

        if (candidate['cat2'] == 3) & (3 in cutoff_dict):
            if (candidate['session1'] >= cutoff_dict[3][0]) & (candidate['session2'] >= cutoff_dict[3][1]):
                cutoff += '3'

        for cat in [4, 5, 7, 8]:
            if (candidate['cat3'] == cat) & (cat in cutoff_dict):
                if (candidate['session1'] >= cutoff_dict[cat][0]) & (candidate['session2'] >= cutoff_dict[cat][1]):
                    cutoff += str(cat)

        return cutoff
    
    filtered_candidates = candidates[(candidates['merit'].notnull())]

    sorted_candidates = filtered_candidates.sort_values(by='merit')

    sorted_candidates[master_record['flag_name']] = sorted_candidates.apply(calculate_cutoff, axis=1)

    candidates.update(sorted_candidates)

    return f"{master_record['flag_name']} updated successfully"

In [23]:
exam_name = 'MTS2024T1'
serial_id = 2

get_cutoff(exam_name, serial_id, master_table_df,cut_off_table_df,candidate_table_df)

'cutoff_flag updated successfully'

In [24]:
candidate_table_df['cutoff_flag'].head(10)

0    91
1      
2    96
3      
4      
5    91
6    90
7      
8      
9      
Name: cutoff_flag, dtype: object

In [25]:
hi = candidate_table_df['cutoff_flag'].value_counts()
hi

cutoff_flag
        1194876
96       347354
9        151463
91       143976
1        116160
         ...   
9635          1
637           1
9135          1
134           1
37            1
Name: count, Length: 79, dtype: int64

In [29]:
candidate_table_df.rename(columns={'arc_code': 'agerelax_code'}, inplace=True)
candidate_table_df['agerelax_code'].unique()





array([ 0,  2,  1, 10, 11,  9, 13, 12,  8,  3,  5,  4,  6])

In [30]:

# candidate_table_df['agerelax_code'] = candidate_table_df['agerelax_code'].fillna(0)
# candidate_table_df['agerelax_code'] = candidate_table_df['agerelax_code'].astype('int')
candidate_table_df['cat1'] = candidate_table_df['cat1'].astype('int')
candidate_table_df['cat2'] = candidate_table_df['cat2'].astype('int')
candidate_table_df['cat3'] = candidate_table_df['cat3'].astype('int')

# hi = candidate_table_df['agerelax_code'].value_counts()
# hi

# hello = candidate_table_df['agerelax_code'] == ''
# hello

In [31]:
candidate_table_df['agerelax_code'].unique()

array([ 0,  2,  1, 10, 11,  9, 13, 12,  8,  3,  5,  4,  6])

### DOB FLAG

In [32]:
mask = candidate_table_df['cat1'].isnull()
candidate_table_df.loc[mask, 'cat1'] = pd.NA
candidate_table_df.loc[~mask, 'cat1'] = candidate_table_df.loc[~mask, 'cat1'].astype('Int64').astype(str)

mask = candidate_table_df['cat2'].isnull()
candidate_table_df.loc[mask, 'cat2'] = pd.NA
candidate_table_df.loc[~mask, 'cat2'] = candidate_table_df.loc[~mask, 'cat2'].astype('Int64').astype(str)

mask = candidate_table_df['cat3'].isnull()
candidate_table_df.loc[mask, 'cat3'] = pd.NA
candidate_table_df.loc[~mask, 'cat3'] = candidate_table_df.loc[~mask, 'cat3'].astype('Int64').astype(str)

# mask = candidate_table_df['agerelax_code'].isnull()
# candidate_table_df.loc[mask, 'agerelax_code'] = ''
# candidate_table_df.loc[~mask, 'agerelax_code'] = candidate_table_df.loc[~mask, 'agerelax_code'].astype(int).astype(str)

mask = candidate_table_df['agerelax_code'].isnull()
candidate_table_df.loc[mask, 'agerelax_code'] = ''
candidate_table_df.loc[~mask, 'agerelax_code'] = candidate_table_df.loc[~mask, 'agerelax_code'].astype(int).astype(str)

# # Convert 'cat1' column
# mask = candidate_table_df['cat1'].isnull()
# candidate_table_df.loc[mask, 'cat1'] = ''
# candidate_table_df.loc[~mask, 'cat1'] = candidate_table_df.loc[~mask, 'cat1'].astype(float).astype(int).astype(str)

# # Convert 'cat2' column
# mask = candidate_table_df['cat2'].isnull()
# candidate_table_df.loc[mask, 'cat2'] = ''
# candidate_table_df.loc[~mask, 'cat2'] = candidate_table_df.loc[~mask, 'cat2'].astype(float).astype(int).astype(str)

# # Convert 'cat3' column (handling empty strings and non-numeric values)
# mask = candidate_table_df['cat3'].isnull()
# candidate_table_df.loc[mask, 'cat3'] = 0  # Replace NaN with 0

# # Ensure only numeric values are converted
# candidate_table_df.loc[candidate_table_df['cat3'] == '', 'cat3'] = 0  # Convert empty strings to 0
# candidate_table_df['cat3'] = candidate_table_df['cat3'].astype(float).astype(int).astype(str)

# # Convert 'arc_code' column
# mask = candidate_table_df['arc_code'].isnull()
# candidate_table_df.loc[mask, 'arc_code'] = ''
# candidate_table_df.loc[~mask, 'arc_code'] = candidate_table_df.loc[~mask, 'arc_code'].astype(float).astype(int).astype(str)

C:\Users\AshutoshMishra\AppData\Local\Temp\ipykernel_12848\2357909575.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['1' '6' '6' ... '1' '6' '1']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  candidate_table_df.loc[~mask, 'cat1'] = candidate_table_df.loc[~mask, 'cat1'].astype('Int64').astype(str)
C:\Users\AshutoshMishra\AppData\Local\Temp\ipykernel_12848\2357909575.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['0' '0' '0' ... '0' '0' '0']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  candidate_table_df.loc[~mask, 'cat2'] = candidate_table_df.loc[~mask, 'cat2'].astype('Int64').astype(str)
C:\Users\AshutoshMishra\AppData\Local\Temp\ipykernel_12848\2357909575.py:11: FutureWarning: Setting an item of incompatible dtype is d

In [85]:
# candidate_table_df[['dob_flag_18_27']] = None


In [33]:
candidate_table_df.rename(columns={'lengthOfService': 'service_period'}, inplace=True)

In [34]:
candidate_table_df[['exsm_yrs', 'exsm_months', 'exsm_days']] = candidate_table_df['service_period'].str.extract(r'(\d+) Year[s]* (\d+) Month[s]* (\d+) Day[s]*')

candidate_table_df[['exsm_yrs', 'exsm_months', 'exsm_days']] = candidate_table_df[['exsm_yrs', 'exsm_months', 'exsm_days']].apply(pd.to_numeric)

In [51]:
# candidate_table_df['dob'].head()
# candidate_table_df['dob'] = pd.to_datetime(candidate_table_df['dob'])



In [35]:
def optimize_get_dob_flag(exam_name, serial_id, master_table, candidate_tbl, arc_table):
    master_record = master_table[(master_table['exam_name'] == exam_name) & (master_table['serial'] == serial_id)].iloc[0]
    
    dob_to_date = pd.to_datetime(master_record['dob_to_date'])
    dob_from_date = pd.to_datetime(master_record['dob_from_date'])

    if serial_id == 4:
        arc_table = arc_table[(arc_table['age_group'] != 'dob_flag_18_25')]

    if serial_id == 3:
        arc_table = arc_table[(arc_table['age_group'] != 'dob_flag_18_27')]
    
    arc_year_dict = dict(zip(arc_table['arc_code'].astype(str), arc_table['arc_year']))
    
    def calculate_verification_flag(row):
        dob = row['dob']
        verification_flag = ''
        
        if dob >= dob_to_date:
            verification_flag = 'U'
        elif dob_from_date < dob < dob_to_date:
            verification_flag = '9'
        elif dob <= dob_from_date:
            agerelax_code = row['agerelax_code']
            
            if pd.isna(agerelax_code) or agerelax_code == '':
                agerelax_code = '99'
            
            if isinstance(agerelax_code, str) and len(agerelax_code.strip()) == 1:
                agerelax_code = agerelax_code.strip()
            
            age_rlx_year = arc_year_dict.get(agerelax_code, 0)
            
            if agerelax_code == '6':
                ex_service_years = row['exsm_yrs']
                ex_service_months = row['exsm_months']
                ex_service_days = row['exsm_days']
                age_rlx_year += int(ex_service_years) if ex_service_years else 0
                new_dob = dob + pd.DateOffset(years=age_rlx_year, months=ex_service_months, days=ex_service_days)
            
            else:
                extract_year = dob.year
                extract_month = dob.month
                extract_day = dob.day
                new_year = extract_year + int(age_rlx_year)
                
                if extract_month == 2 and extract_day == 29:
                    extract_day = 28
            
                new_dob = pd.to_datetime(f"{new_year}-{extract_month}-{extract_day}", format='%Y-%m-%d')
                
            if new_dob > dob_from_date:
                if len(agerelax_code) < 2:
                    verification_flag = '0' + agerelax_code
                else:
                    verification_flag = agerelax_code
            else:
                verification_flag = '99'

        return verification_flag

    filtered_candidates = candidate_tbl[(candidate_tbl['merit'].notnull())]
    sorted_candidates = filtered_candidates.sort_values(by='merit')

    sorted_candidates[master_record['flag_name']] = sorted_candidates.apply(calculate_verification_flag, axis=1)

    candidate_tbl.update(sorted_candidates)
    
    return f"{master_record['flag_name']} updated successfully"


In [36]:
exam_name = 'MTS2024T1'
serial_id = 4

optimize_get_dob_flag(exam_name, serial_id, master_table_df, candidate_table_df, arc_table_df)

'dob_flag_18_27 updated successfully'

In [37]:
candidate_table_df['dob_flag_18_27'].head(10)

# import numpy as np

# candidate_table_df.drop(columns=['dob_flag_18_27'], inplace=True)



0    9
1    9
2    9
3    9
4    9
5    9
6    9
7    9
8    9
9    9
Name: dob_flag_18_27, dtype: object

In [38]:
candidate_table_df['dob_flag_18_27'].unique()

array(['9', '02', '01', '11', '13', '10', '03', '05', '04', '08', '09',
       '06', '12', '99', None], dtype=object)

In [91]:
# candidate_table_df['dob_flag_18_27'].unique()

array(['9', '02', '01', '11', '13', '10', '03', '05', '04', '08', '09',
       '06', '12', '99', None], dtype=object)

In [11]:
# candidate_table_df['dob_flag_18_27'].unique()

array([ 9.,  2.,  1., 11., 13., 10.,  3.,  5.,  4.,  8.,  6., 12., 99.,
       nan])

In [39]:
null_cutoff_rows = candidate_table_df[

    candidate_table_df['dob_flag_18_27'] == '99'

][['dob', 'cat1', "cat2", "cat3", 'merit', "agerelax_code","rej_prov","debarred"]].head(10)
 
 
print(null_cutoff_rows)
 

             dob cat1 cat2 cat3    merit agerelax_code rej_prov debarred
2929  1981-06-05    9    3    0  2153078             8      NaN      NaN
2951  1983-05-13    6    3    0  1133127             2      NaN      NaN
9241  1997-04-13    9    0    0  1829681             0      NaN      NaN
12891 1977-07-05    9    3    0  2248859             8      NaN      NaN
15888 1992-06-18    1    0    0  1123173             1      NaN      NaN
16834 1992-04-24    6    0    0  1811859             2      NaN      NaN
22202 1983-07-07    1    3    0  1617090             1      NaN      NaN
31268 1982-03-05    9    3    0   983823             8      NaN      NaN
31277 1984-04-21    6    3    0  1420091             2      NaN      NaN
32013 1993-12-17    6    0    0   456533             2      NaN      NaN


In [40]:
exam_name = 'MTS2024T1'
serial_id = 3

optimize_get_dob_flag(exam_name, serial_id, master_table_df, candidate_table_df, arc_table_df)

'dob_flag_18_25 updated successfully'

In [41]:
candidate_table_df[['rollno', 'dob_flag_18_25']].head(20)

,rollno,dob_flag_18_25
0,3206058346,99
1,6016010490,9
2,3206243907,9
3,6006043049,9
4,3010175465,9
5,3010035576,9
6,8007037382,9
7,1004019963,9
8,8603004453,9
9,3013183689,99


In [42]:
candidate_table_df['dob_flag_18_25'].unique()


null_cutoff_rows = candidate_table_df[

    candidate_table_df['dob_flag_18_25'] == '99'

][['dob', 'cat1', "cat2", "cat3", 'merit', "agerelax_code","rej_prov","debarred"]].head(10)
 
 
print(null_cutoff_rows)

           dob cat1 cat2 cat3    merit agerelax_code rej_prov debarred
0   1999-02-12    1    0    0   744253             0      NaN      NaN
9   1997-11-23    6    0    0  1975103             0      NaN      NaN
42  1996-06-06    6    0    0   389222             2      NaN      NaN
48  1998-01-20    9    0    0   493981             0      NaN      NaN
56  1998-05-03    9    0    0   873142             0      NaN      NaN
72  1998-12-07    0    0    0   860490             0      NaN      NaN
78  1998-02-23    9    0    0   815200             0      NaN      NaN
99  1997-10-09    6    0    0  1626985             0      NaN      NaN
106 1997-09-11    6    0    0  1348300             0      NaN      NaN
112 1999-04-05    1    0    0  2033918             0      NaN      NaN


### CATSEL DOB FLAG

In [43]:
arc_table_df['arc_code'] = arc_table_df['arc_code'].astype(str)

mask = arc_table_df['arc_cat2'].isnull()
arc_table_df.loc[mask, 'arc_cat2'] = ''

mask = arc_table_df['arc_cat3'].isnull()
arc_table_df.loc[mask, 'arc_cat3'] = ''

In [44]:
arc_table_df['arc_code'] = arc_table_df['arc_code'].apply(lambda x: str(x).zfill(2))

In [45]:
def get_catsel_dob(exam_name, serial_id, master_table, candidate_table, arc_table):
    master_record = master_table[(master_table['exam_name'] == exam_name) & (master_table['serial'] == serial_id)].iloc[0]
    dob_to_date = pd.to_datetime(master_record['dob_to_date'])
    dob_from_date = pd.to_datetime(master_record['dob_from_date'])
    
    if serial_id == 6:
        arc_table = arc_table[(arc_table['age_group'] != 'dob_flag_18_25')]
        dob_flag = 'dob_flag_18_27'
   
    elif serial_id == 5:
        arc_table = arc_table[(arc_table['age_group'] != 'dob_flag_18_27')]
        dob_flag = 'dob_flag_18_25'
        
    filtered_and_sorted_candidates = candidate_table[candidate_table['merit'].notnull()].sort_values(by='merit')
    
    def calculate_catsel_dob(row, dob_flag_name):

        dob_flag = row[dob_flag_name]
       
        if dob_flag is not None and dob_flag != '' and dob_flag != '9':
            matching_rows = arc_table[arc_table['arc_code'] == dob_flag]
            if not matching_rows.empty:
                catsel = matching_rows.iloc[0]
                # print(row[['cat1', 'cat3', 'dob_flag_18_27']])
                catsel_cat1 = catsel['arc_cat1'].split(',')
                catsel_cat2 = catsel['arc_cat2'].split(',')
                catsel_cat3 = catsel['arc_cat3'].split(',')
                catsel_gender = catsel['arc_gender'].split(',')
                condition = (str(row['cat1']) in catsel_cat1) & (str(int(row['gender'])) in catsel_gender)
                catsel_dob = ''
                if row['cat2'] == '3' and row['cat2'] in catsel_cat2:
                    catsel_dob = row['cat2']
                else:
                    if pd.isnull(catsel['arc_year_against_ur']):
                        catsel['arc_year_against_ur'] = 0
                    dob_datetime = pd.to_datetime(row['dob'])
                    extract_year = dob_datetime.year
                    extract_month = dob_datetime.month
                    extract_day = dob_datetime.day
                    if (extract_day == 29) and (extract_month == 2):
                        extract_day = 28
                    new_year = extract_year + int(catsel['arc_year_against_ur'])
                    new_dob = pd.to_datetime(f'{new_year}-{extract_month}-{extract_day}')
                    catsel_dob_decision = new_dob > dob_from_date
                    if (str(row['cat3']) in catsel_cat3 and str(row['cat3']) != ''):
                        if catsel_dob_decision:
                            catsel_dob = '9'
                        else:
                            catsel_dob = row['cat3']  
                    else:
                        catsel_dob = '9' if catsel_dob_decision else row['cat1']
                return catsel_dob if condition else ''
            else:
                return ''
        else:
            return dob_flag
    
    filtered_and_sorted_candidates[master_record['flag_name']] = filtered_and_sorted_candidates.apply(calculate_catsel_dob, args = (dob_flag, ), axis=1)
    candidate_table.update(filtered_and_sorted_candidates)
    
    print('Successfully updated catsel_dob')

In [46]:
exam_name = 'MTS2024T1'
serial_id = 6

get_catsel_dob(exam_name, serial_id, master_table_df, candidate_table_df, arc_table_df)

Successfully updated catsel_dob


In [47]:
candidate_table_df[['rollno', 'catsel_dob_18_27']].head(20)

,rollno,catsel_dob_18_27
0,3206058346,9
1,6016010490,9
2,3206243907,9
3,6006043049,9
4,3010175465,9
5,3010035576,9
6,8007037382,9
7,1004019963,9
8,8603004453,9
9,3013183689,9


In [48]:
candidate_table_df['catsel_dob_18_27'].unique()

array(['9', '6', '2', '1', '4', '3', '', '8', '7', '5', None],
      dtype=object)

In [49]:
exam_name = 'MTS2024T1'
serial_id = 5
get_catsel_dob(exam_name, serial_id, master_table_df, candidate_table_df, arc_table_df)

Successfully updated catsel_dob


In [50]:
candidate_table_df[['rollno', 'catsel_dob_18_25']].head(20)

,rollno,catsel_dob_18_25
0,3206058346,
1,6016010490,9
2,3206243907,9
3,6006043049,9
4,3010175465,9
5,3010035576,9
6,8007037382,9
7,1004019963,9
8,8603004453,9
9,3013183689,


In [51]:
candidate_table_df['catsel_dob_18_25'].unique()

array(['', '9', '6', '2', '1', '3', '7', '4', '8', '5', None],
      dtype=object)

### CATSEL FLAG

In [52]:
def get_catsel(par_exam_name, par_serialid, master_table_df, candidates_df):
    master_record = master_table_df[(master_table_df['exam_name'] == par_exam_name) & (master_table_df['serial'] == par_serialid)].iloc[0]
    
    catsel_dob = 'catsel_dob_' + master_record['flag_name'].strip('catsel_')
    
    candidates = candidates_df[(candidates_df['merit'].notnull()) &
                               (candidates_df[catsel_dob].notna()) &
                               (candidates_df[catsel_dob] != '')].sort_values(by='rollno')
    
    def update_cutoff_flag(candidate_record):
        if candidate_record['cat2'] == '3' and candidate_record['exs_reservation'] == 'No':
            candidate_record['cutoff_flag'] = candidate_record['cutoff_flag'].replace('3', '').strip()

        if candidate_record[catsel_dob] in ['1', '2', '6', '4', '5', '7', '8']:
            return candidate_record['cutoff_flag'].replace('9', '').strip()
        elif pd.isna(candidate_record[catsel_dob]) or candidate_record[catsel_dob] == '':
            return ''
        else:
            return candidate_record['cutoff_flag']
    
    candidates_df[master_record['flag_name']] = candidates.apply(update_cutoff_flag, axis=1)

    return 'Successfully updated catsel'

In [53]:
exam_name = 'MTS2024T1'
serial_id = 8

get_catsel(exam_name, serial_id, master_table_df, candidate_table_df)

'Successfully updated catsel'

In [54]:
candidate_table_df['catsel_18_27'].head(10)

0    91
1      
2    96
3      
4      
5    91
6    90
7      
8      
9      
Name: catsel_18_27, dtype: object

In [55]:
# candidate_table_df['catsel_18_27'].unique()

# candidate_table_df['dob_flag_18_25'].unique()


# null_cutoff_rows = candidate_table_df[

#     candidate_table_df['exs_reservation'] == 'No'

# ][['dob', 'cat1', "cat2", "cat3", 'merit', "agerelax_code","rej_prov","debarred","catsel_18_27","catsel_18_25","cutoff_flag"]].head(10)
 
 
# print(null_cutoff_rows)

In [56]:
exam_name = 'MTS2024T1'
serial_id = 7

get_catsel(exam_name, serial_id, master_table_df, candidate_table_df)

'Successfully updated catsel'

In [57]:
candidate_table_df['catsel_18_25'].head(10)

0    NaN
1       
2     96
3       
4       
5     91
6     90
7       
8       
9    NaN
Name: catsel_18_25, dtype: object

### Copying both candidates for temp measures

In [58]:
origin_vacancy = vacancy_table_df.copy()

In [59]:
vacancy_table_df

,post_name,post_code,state_code,state,region,age_limit,category,category_code,original_vacancy,initial_vacancy,current,allocated,left_vacancy,key,min_marks_prev,minpart4marks_prev,lowestmarkssession1_prev,min_marks_cand_dob_prev
0,HAVALDAR-CGST,HCG,11,Chandigarh- Hawaldar,NWR,18-27,EWS,0,11,11,11,NaN,NaN,01118-27,126.26082,35.0,55.58734,2006-06-12
1,HAVALDAR-CGST,HCG,11,Chandigarh- Hawaldar,NWR,18-27,SC,1,10,10,10,NaN,NaN,11118-27,121.96142,31.0,66.46335,2005-09-18
2,HAVALDAR-CGST,HCG,11,Chandigarh- Hawaldar,NWR,18-27,ST,2,2,2,2,NaN,NaN,21118-27,122.14574,35.0,74.60975,2006-05-01
3,HAVALDAR-CGST,HCG,11,Chandigarh- Hawaldar,NWR,18-27,ESM,3,8,8,8,NaN,NaN,31118-27,90.18447,10.0,44.23924,1992-11-18
4,HAVALDAR-CGST,HCG,11,Chandigarh- Hawaldar,NWR,18-27,OH,4,1,1,1,NaN,NaN,41118-27,122.72300,36.0,105.24359,2001-07-26
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
778,MTS,MTS,68,Kerala,KKR,18-27,HH,5,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
779,MTS,MTS,68,Kerala,KKR,18-27,OBC,6,7,7,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
780,MTS,MTS,68,Kerala,KKR,18-27,VH,7,1,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
781,MTS,MTS,68,Kerala,KKR,18-27,PWD-Others,8,2,2,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [60]:
cand = candidate_table_df.copy()

In [88]:
# candidate_table_df.head()

In [89]:
# import pandas as pd

# # Excel file ka naam
# output_file = "candidate_table_Final_for_Processing.xlsx"

# # Ek sheet me max 10 lakh (1 million) rows dalenge
# chunk_size = 1000000  

# with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
#     for i in range(0, len(candidate_table_df), chunk_size):
#         # Chunk ko alag sheet me save karein
#         candidate_table_df.iloc[i:i+chunk_size].to_excel(
#             writer, sheet_name=f'Sheet_{i//chunk_size+1}', index=False
#         )

# print("Excel file save ho gaya! ✅")


In [3]:
# candidate_table_df = pd.read_csv("candidate_table_Final_for_Processing.csv")

C:\Users\AshutoshMishra\AppData\Local\Temp\ipykernel_7340\684697010.py:1: DtypeWarning: Columns (7,35,39,40,41,42,43,53,59,60,84,85,99,100,106,109,111,112,113,114,115,116,118,119,121,122,125,127,128,129,130,131) have mixed types. Specify dtype option on import or set low_memory=False.
  candidate_table_df = pd.read_csv("candidate_table_Final_for_Processing.csv")


In [61]:
candidate_table_df['pt_finalstatus'].unique()

array([nan, 'Not Qualified', 'Qualified', 'With Held', 'Not Qualfied',
       'Temporary Unfit'], dtype=object)

### Allocation

#### Prerequisites

In [37]:
'''vacancy_table_df['state_code'] = vacancy_table_df['state_code'].astype(str)
vacancy_table_df['category_code'] = vacancy_table_df['category_code'].astype(str)
vacancy_table_df['allocated'] = 0'''

"vacancy_table_df['state_code'] = vacancy_table_df['state_code'].astype(str)\nvacancy_table_df['category_code'] = vacancy_table_df['category_code'].astype(str)\nvacancy_table_df['allocated'] = 0"

In [38]:
#vacancy_table_df['key'] = vacancy_table_df['category_code'] + vacancy_table_df['state_code'] + '18-27'

In [39]:
mask = candidate_table_df['catsel_18_27'].isnull()
candidate_table_df.loc[mask, 'catsel_18_27'] = ''
candidate_table_df.loc[~mask, 'catsel_18_27'] = candidate_table_df.loc[~mask, 'catsel_18_27'].astype(str)

In [40]:
'''candidate_table_df['state_pref_havaldar'] = ''
state_list_set = set(vacancy_table_df['state_code'].unique())

def filter_states(state_pref):
    if pd.notna(state_pref):
        states = state_pref.split(',')
        filtered_states = ','.join(state.strip() for state in states if state.strip() in state_list_set)
        return filtered_states
    return state_pref

candidate_table_df['state_pref_havaldar'] = candidate_table_df['state_ut_pref'].apply(filter_states)'''

"candidate_table_df['state_pref_havaldar'] = ''\nstate_list_set = set(vacancy_table_df['state_code'].unique())\n\ndef filter_states(state_pref):\n    if pd.notna(state_pref):\n        states = state_pref.split(',')\n        filtered_states = ','.join(state.strip() for state in states if state.strip() in state_list_set)\n        return filtered_states\n    return state_pref\n\ncandidate_table_df['state_pref_havaldar'] = candidate_table_df['state_ut_pref'].apply(filter_states)"

In [41]:
def allocate_candidates(candidates_df, vacancy_df):
    result_messages = []
    vacancy_map_18_27 = {}

    filtered_candidates = candidates_df[
        (candidates_df['merit'].notnull()) & ((candidates_df['cat3'].isnull()) | (candidates_df['cat3'] != '7'))
    ].sort_values(by='merit')

    for _, vacancy in vacancy_df.iterrows():
        key = f"{vacancy['category_code']}{vacancy['state_code']}18-27"
        vacancy_map_18_27[key] = {
            'current': vacancy['current'],
            'allocated': 0,
            'initial': vacancy['initial_vacancy']
        }

    for _, candidate in filtered_candidates.iterrows():
        
        state_preference = candidate['state_pref_havaldar']
        
        allocated = False
        
        if pd.notna(state_preference):
            
            state_list = [state.strip() for state in state_preference.split(',') if state.strip() != 'X']
            
            for state in state_list:
                
                catsel_18_27 = candidate['catsel_18_27']
                roll = candidate['rollno']
                cat1 = candidate['cat1']
                for category in catsel_18_27:
                    category = str(category)
                    
                    key = f"{category}{state}18-27"

                    vacancy_18_27 = vacancy_map_18_27.get(key, {'current': 0, 'allocated': 0, 'initial': 0})
                    
                    if vacancy_18_27['current'] > 0:
                        allocated_against_ur = ""
                        
                        '''if category in ['3', '4', '5', '7', '8']:
                            vacancy_key_cat1 = f"{cat1}{state}18-27"
                            vacancy_key_9 = f"9{state}18-27"

                            vacancy_18_27_cat1 = vacancy_map_18_27.get(vacancy_key_cat1, {'initial': 0})
                            vacancy_18_27_cat2 = vacancy_map_18_27.get(vacancy_key_9, {'initial': 0})

                            if vacancy_18_27_cat1['initial'] == 0 and vacancy_18_27_cat2['initial'] == 0:
                                continue
                            elif vacancy_18_27_cat2['initial'] > 0:
                                allocated_against_ur = "1"'''

                        candidates_df.loc[_, 'age_limit_havaldar'] = '18-27'
                        candidates_df.loc[_, 'allocated_category_havaldar'] = category
                        candidates_df.loc[_, 'allocated_state_havaldar'] = state
                        candidates_df.loc[_, 'allocated_against_ur_havaldar'] = allocated_against_ur
                        
                        allocated = True

                        
                        result_messages.append(f"Candidate roll: {roll} category: {category} state: {state}")
                        
                        
                        if allocated:
                            vacancy_18_27['current'] -= 1
                            vacancy_18_27['allocated'] += 1

                        if allocated:
                            break
                if allocated:
                    break
    return result_messages, vacancy_map_18_27

In [42]:
#result_msg, vacancy_dict = allocate_candidates(candidate_table_df, vacancy_table_df)

#### Updating VacancyTable

In [43]:
'''vacancy_df = pd.DataFrame.from_dict(vacancy_dict, orient='index')

for index in vacancy_df.index:
    if index in vacancy_table_df['key']:
        vacancy_table_df.loc[index == vacancy_table_df['key'], 'current'] = vacancy_df.loc[index, 'current']
        vacancy_table_df.loc[index == vacancy_table_df['key'], 'allocated'] = vacancy_df.loc[index, 'allocated']
        vacancy_table_df.loc[index == vacancy_table_df['key'], 'initial_vacancy'] = vacancy_df.loc[index, 'initial']'''


"vacancy_df = pd.DataFrame.from_dict(vacancy_dict, orient='index')\n\nfor index in vacancy_df.index:\n    if index in vacancy_table_df['key']:\n        vacancy_table_df.loc[index == vacancy_table_df['key'], 'current'] = vacancy_df.loc[index, 'current']\n        vacancy_table_df.loc[index == vacancy_table_df['key'], 'allocated'] = vacancy_df.loc[index, 'allocated']\n        vacancy_table_df.loc[index == vacancy_table_df['key'], 'initial_vacancy'] = vacancy_df.loc[index, 'initial']"

#### Verifications of Allocation

In [44]:
'''allocated_candidates = candidate_table_df[candidate_table_df['allocated_category_havaldar'].notnull()]
allocated_candidates.shape'''

"allocated_candidates = candidate_table_df[candidate_table_df['allocated_category_havaldar'].notnull()]\nallocated_candidates.shape"

#### Writing to CSV

In [45]:
#allocated_candidates.to_csv('only_allocated_candidates.csv', index = False)

In [46]:
#candidate_table_df.to_csv('allocaton_completed.csv', index = False)

### Allocation Tier 2

In [47]:
# cand = candidate_table_df.copy()

In [48]:
# candidate_table_df = cand.copy()

In [62]:
mask = candidate_table_df['cat1'].isnull()
candidate_table_df.loc[mask, 'cat1'] = pd.NA
candidate_table_df.loc[~mask, 'cat1'] = candidate_table_df.loc[~mask, 'cat1'].astype(int).astype(str)

mask = candidate_table_df['cat2'].isnull()
candidate_table_df.loc[mask, 'cat2'] = pd.NA
candidate_table_df.loc[~mask, 'cat2'] = candidate_table_df.loc[~mask, 'cat2'].astype(int).astype(str)

mask = candidate_table_df['cat3'].isnull()
candidate_table_df.loc[mask, 'cat3'] = pd.NA
candidate_table_df.loc[~mask, 'cat3'] = candidate_table_df.loc[~mask, 'cat3'].astype(int).astype(str)

mask = candidate_table_df['catsel_18_25'].isnull()
candidate_table_df.loc[mask, 'catsel_18_25'] = ''
candidate_table_df.loc[~mask, 'catsel_18_25'] = candidate_table_df.loc[~mask, 'catsel_18_25'].astype(str)

mask = candidate_table_df['catsel_18_27'].isnull()
candidate_table_df.loc[mask, 'catsel_18_27'] = ''
candidate_table_df.loc[~mask, 'catsel_18_27'] = candidate_table_df.loc[~mask, 'catsel_18_27'].astype(str)

mask = candidate_table_df['catsel_dob_18_27'].isnull()
candidate_table_df.loc[mask, 'catsel_dob_18_27'] = ''
candidate_table_df.loc[~mask, 'catsel_dob_18_27'] = candidate_table_df.loc[~mask, 'catsel_dob_18_27'].astype(str)



# # Handling 'cat1' column
# mask = candidate_table_df['cat1'].isnull()
# candidate_table_df.loc[mask, 'cat1'] = ''
# candidate_table_df.loc[~mask, 'cat1'] = candidate_table_df.loc[~mask, 'cat1'].replace('', 0).astype(int).astype(str)

# # Handling 'cat2' column
# mask = candidate_table_df['cat2'].isnull()
# candidate_table_df.loc[mask, 'cat2'] = ''
# candidate_table_df.loc[~mask, 'cat2'] = candidate_table_df.loc[~mask, 'cat2'].replace('', 0).astype(int).astype(str)

# # Handling 'cat3' column
# mask = candidate_table_df['cat3'].isnull()
# candidate_table_df.loc[mask, 'cat3'] = ''
# candidate_table_df.loc[~mask, 'cat3'] = candidate_table_df.loc[~mask, 'cat3'].replace('', 0).astype(int).astype(str)

# # Handling 'catsel_18_25' column
# mask = candidate_table_df['catsel_18_25'].isnull()
# candidate_table_df.loc[mask, 'catsel_18_25'] = ''
# candidate_table_df.loc[~mask, 'catsel_18_25'] = candidate_table_df.loc[~mask, 'catsel_18_25'].replace('', 0).astype(int).astype(str)

# # Handling 'catsel_18_27' column
# mask = candidate_table_df['catsel_18_27'].isnull()
# candidate_table_df.loc[mask, 'catsel_18_27'] = ''
# candidate_table_df.loc[~mask, 'catsel_18_27'] = candidate_table_df.loc[~mask, 'catsel_18_27'].replace('', 0).astype(int).astype(str)

# # Handling 'catsel_dob_18_27' column
# mask = candidate_table_df['catsel_dob_18_27'].isnull()
# candidate_table_df.loc[mask, 'catsel_dob_18_27'] = ''
# candidate_table_df.loc[~mask, 'catsel_dob_18_27'] = candidate_table_df.loc[~mask, 'catsel_dob_18_27'].replace('', 0).astype(int).astype(str)




In [63]:
candidate_table_df['catsel_18_25'].unique()

array(['', '96', '91', '90', '1', '9', '0', '6', '2', '967', '92', '14',
       '964', '95', '97', '17', '7', '3', '13', '138', '98', '4', '917',
       '963', '5', '24', '64', '63', '68', '8', '968', '914', '94', '93',
       '913', '65', '67', '927', '18', '918', '924', '27', '965', '923',
       '904', '907', '04', '23', '905', '15', '903', '28', '908', '05',
       '634', '9634', '25', '928', '938', '38', '34', '915', '934', '935',
       '9034', '03', '925', '638', '35', '07', '9638', '08', '035',
       '9135', '134', '937', '37'], dtype=object)

In [92]:
# candidate_table_df['catsel_18_25'].unique()
# candidate_table_df['catsel_18_25'] = candidate_table_df['catsel_18_25'].astype(float).astype(int).astype(str)

# candidate_table_df['catsel_18_27'].unique()
# candidate_table_df['catsel_dob_18_27'].unique()

array(['', '96', '91', '90', '1', '9', '0', '6', '2', '967', '92', '14',
       '964', '95', '97', '17', '7', '3', '13', '138', '98', '4', '917',
       '963', '5', '24', '64', '63', '68', '914', '8', '968', '94', '93',
       '913', '65', '67', '927', '18', '918', '924', '27', '965', '923',
       '904', '907', '04', '23', '905', '15', '903', '28', '908', '05',
       '634', '9634', '25', '928', '938', '38', '34', '915', '934', '935',
       '9034', '03', '925', '638', '35', '07', '9638', '08', '035',
       '9135', '134', '937', '37'], dtype=object)

In [64]:
mask = candidate_table_df['post_pref'].isnull()
candidate_table_df.loc[mask, 'post_pref'] = ''
candidate_table_df.loc[~mask, 'post_pref'] = candidate_table_df.loc[~mask, 'post_pref'].astype(str)

In [65]:
vacancy_df = pd.read_csv(r"C:\Users\AshutoshMishra\OneDrive - Cubastion Consulting Pvt Ltd\Desktop\SSC OPS\RESULT PROCESSING MTS 2024\mts 2024\vacancy_table_for_result_processing.csv")
vacancy_df['allocated_hc'] = 0

In [66]:
vacancy_df

,post_name,post_code,state_code,state,region,age_limit,category,category_code,original_vacancy,initial_vacancy,current,allocated,left_vacancy,key,min_marks_prev,minpart4marks_prev,lowestmarkssession1_prev,min_marks_cand_dob_prev,allocated_hc
0,HAVALDAR-CGST,HCG,11,Chandigarh- Hawaldar,NWR,18-27,EWS,0,11,11,11,NaN,NaN,01118-27,126.26082,35.0,55.58734,2006-06-12,0
1,HAVALDAR-CGST,HCG,11,Chandigarh- Hawaldar,NWR,18-27,SC,1,10,10,10,NaN,NaN,11118-27,121.96142,31.0,66.46335,2005-09-18,0
2,HAVALDAR-CGST,HCG,11,Chandigarh- Hawaldar,NWR,18-27,ST,2,2,2,2,NaN,NaN,21118-27,122.14574,35.0,74.60975,2006-05-01,0
3,HAVALDAR-CGST,HCG,11,Chandigarh- Hawaldar,NWR,18-27,ESM,3,8,8,8,NaN,NaN,31118-27,90.18447,10.0,44.23924,1992-11-18,0
4,HAVALDAR-CGST,HCG,11,Chandigarh- Hawaldar,NWR,18-27,OH,4,1,1,1,NaN,NaN,41118-27,122.72300,36.0,105.24359,2001-07-26,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
778,MTS,MTS,68,Kerala,KKR,18-27,HH,5,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
779,MTS,MTS,68,Kerala,KKR,18-27,OBC,6,7,7,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
780,MTS,MTS,68,Kerala,KKR,18-27,VH,7,1,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
781,MTS,MTS,68,Kerala,KKR,18-27,PWD-Others,8,2,2,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [67]:
havaldar_state_set = set(vacancy_df.loc[vacancy_df['post_code'] != 'MTS', 'state_code'].astype(str))
havaldar_state_lst = list(havaldar_state_set)

In [68]:
vacancy_df['left_vacancy'] = vacancy_df['current']
vacancy_dict = {}
vacancy_df['key'] = vacancy_df['category_code'].astype(str) + vacancy_df['state_code'].astype(str) + vacancy_df['age_limit'].astype(str)

for index, row in vacancy_df.iterrows():
    key = row['key']
    vacancy_dict[key] = row.to_dict()

In [69]:
alloc_cand = candidate_table_df.copy()

In [70]:
alloc_cand[['age_limit', 'allocated_category', 'allocated_state', 'allocated_against_ur']] = None

In [71]:
def update_allocation_18_25(candidates_df, category, state, allocated_against_ur, candidate, vacancy_dict):
    if str(category) in ['3', '4', '5', '7', '8']:
        key_cat1 = str(candidate['cat1']) + str(state) + '18-25'
        
        if (key_cat1 not in vacancy_dict.keys()) or (vacancy_dict[key_cat1]['initial_vacancy'] == 0):
            keyCat2='9'+str(state)+'18-25'
            
            if (keyCat2 not in vacancy_dict.keys()) or (vacancy_dict[keyCat2]['initial_vacancy'] == 0):
                return False
            else:
                if vacancy_dict[keyCat2]['allocated_hc'] != vacancy_dict[keyCat2]['initial_vacancy']:
                    vacancy_dict[keyCat2]['allocated_hc'] += 1
                allocated_against_ur = '1'
        else:
            if vacancy_dict[key_cat1]['allocated_hc'] != vacancy_dict[key_cat1]['initial_vacancy']:
                vacancy_dict[key_cat1]['allocated_hc'] += 1
            
            else :
                return False
                        
    candidates_df.loc[candidate.name, 'age_limit'] = '18-25'
    candidates_df.loc[candidate.name, 'allocated_category'] = category
    candidates_df.loc[candidate.name, 'allocated_state'] = state
    candidates_df.loc[candidate.name, 'allocated_against_ur'] = allocated_against_ur
    
    return True

In [72]:
def update_allocation_18_27(candidates_df, category, state, allocated_against_ur, candidate, vacancy_dict):
    
    if str(category) in ['3', '4', '5', '7', '8']:
        key_cat1 = str(candidate['cat1']) + str(state) + '18-27'
        
        if (key_cat1 not in vacancy_dict.keys()) or (vacancy_dict[key_cat1]['initial_vacancy'] == 0):
            keyCat2='9'+str(state)+'18-27'
    
            if (keyCat2 not in vacancy_dict.keys()) or (vacancy_dict[keyCat2]['initial_vacancy'] == 0):
                return False
            else:
                if vacancy_dict[keyCat2]['allocated_hc'] != vacancy_dict[keyCat2]['initial_vacancy']:
                    vacancy_dict[keyCat2]['allocated_hc'] += 1
                allocated_against_ur = '1'
        else:
            if vacancy_dict[key_cat1]['allocated_hc'] != vacancy_dict[key_cat1]['initial_vacancy']:
                vacancy_dict[key_cat1]['allocated_hc'] += 1
            else :
                return False
                
    candidates_df.loc[candidate.name, 'age_limit'] = '18-27'
    candidates_df.loc[candidate.name, 'allocated_category'] = category
    candidates_df.loc[candidate.name, 'allocated_state'] = state
    candidates_df.loc[candidate.name, 'allocated_against_ur'] = allocated_against_ur

    return True

In [73]:
def update_allocation_18_27_havaldar(candidates_df, category, state, candidate, vacancy_dict):
    key = str(category) + str(state) + '18-27'
    eligible = False
    flag = False
    allocated_against_ur = ''

    if key in vacancy_dict and vacancy_dict[key]['current'] > 0:
        marks_conditions_met = (
            candidate['session2'] > vacancy_dict[key]['min_marks_prev'] or
            (candidate['session2'] == vacancy_dict[key]['min_marks_prev'] and
             candidate['part3_ga'] > vacancy_dict[key]['minpart4marks_prev']) or
            (candidate['session2'] == vacancy_dict[key]['min_marks_prev'] and
             candidate['part3_ga'] == vacancy_dict[key]['minpart4marks_prev'] and
             candidate['session1'] > vacancy_dict[key]['lowestmarkssession1_prev']) or
            (candidate['session2'] == vacancy_dict[key]['min_marks_prev'] and
             candidate['part3_ga'] == vacancy_dict[key]['minpart4marks_prev'] and
             candidate['session1'] == vacancy_dict[key]['lowestmarkssession1_prev'] and
             pd.to_datetime(candidate['dob']) <= pd.to_datetime(vacancy_dict[key]['min_marks_cand_dob_prev']))
        )

        if marks_conditions_met:
            if str(category) in ['3', '4', '5', '7', '8']:
                key_cat1 = str(candidate['cat1']) + str(state) + '18-27'

                if key_cat1 in vacancy_dict and vacancy_dict[key_cat1]['initial_vacancy'] == 0:
                    if str(category) == '3' or (str(category) in ['4', '5', '7', '8'] and candidate['catsel_dob_18_27'] == '9'):
                        keyCat2 = '9' + str(state) + '18-27'

                        if keyCat2 in vacancy_dict and vacancy_dict[keyCat2]['initial_vacancy'] == 0:
                            flag = False
                        else:
                            allocated_against_ur = '1'
                            if vacancy_dict[keyCat2]['allocated_hc'] != vacancy_dict[keyCat2]['initial_vacancy']:
                                vacancy_dict[keyCat2]['allocated_hc'] += 1
                            flag = True
            
                elif key_cat1 in vacancy_dict:
                    if vacancy_dict[key_cat1]['allocated_hc'] != vacancy_dict[key_cat1]['initial_vacancy']:
                        vacancy_dict[key_cat1]['allocated_hc'] += 1
                        flag = True
                    else :
                        flag = False
            else:
                flag = True

            if flag:
                candidates_df.loc[candidate.name, 'age_limit'] = '18-27'
                candidates_df.loc[candidate.name, 'allocated_category'] = category
                candidates_df.loc[candidate.name, 'allocated_state'] = state
                candidates_df.loc[candidate.name, 'allocated_against_ur'] = allocated_against_ur
                vacancy_dict[key]['current'] -= 1
                vacancy_dict[key]['allocated'] += 1

    return flag


In [74]:
def allocate_candidates(candidates_df, vacancy_dict, havaldar_state_lst):
    allocated_count = 0
    
    candidates_df['dob'] = pd.to_datetime(candidates_df['dob'])

    filtered_candidates = candidates_df[((candidates_df['cutoff_flag'].notnull()) &
                                         (candidates_df['cutoff_flag'] != '') &
                                         (candidates_df['merit'].notnull()))].sort_values(by='merit')

    
    for idx, candidate in filtered_candidates.iterrows():
        allocated = False
        roll = candidate['rollno']
        state_preference = candidate['post_pref']
        DOB = candidate['dob']
        
        for state in state_preference.split(','):
            if state == 'X':
                continue

            catsel_18_25 = str(candidate['catsel_18_25'])
            catsel_18_27 = str(candidate['catsel_18_27'])
            
            if catsel_18_25 != "":
                for category in catsel_18_25:
                    key_18_25 = str(category) + str(state) + '18-25'
                    allocated_against_ur = ''
                        
                    if key_18_25 in vacancy_dict and vacancy_dict[key_18_25]['current'] > 0:
                                    
                        allocated = update_allocation_18_25(candidates_df, category, state, allocated_against_ur, candidate, vacancy_dict)
                        if allocated:
                            vacancy_dict[key_18_25]['current'] -= 1
                            vacancy_dict[key_18_25]['allocated'] += 1
                            allocated_count += 1
                            break
                if allocated:
                    break

            if catsel_18_27 != "":
                if state in havaldar_state_lst:
                    if candidate['pt_finalstatus'] in ['Qualified', 'With Held','Temporary Unfit']:
                        for category in catsel_18_27:
                            allocated = update_allocation_18_27_havaldar(candidates_df, category, state, candidate, vacancy_dict)
                            if allocated:
                                break
                        if allocated:
                            break
                
                else:
                    for category in catsel_18_27:
                        key_18_27 = str(category) + str(state) + '18-27'
                        allocated_against_ur = ''
                        
                        if key_18_27 in vacancy_dict and vacancy_dict[key_18_27]['current'] > 0:            
                            allocated = update_allocation_18_27(candidates_df, category, state, allocated_against_ur, candidate, vacancy_dict)
                            
                            if allocated:
                                vacancy_dict[key_18_27]['current'] -= 1
                                vacancy_dict[key_18_27]['allocated'] += 1
                                
                                break
                    if allocated:
                        break

        if allocated_count == len(candidates_df):
            break

    return vacancy_dict

In [75]:
upd_vacancy_dict = allocate_candidates(alloc_cand, vacancy_dict, havaldar_state_lst)

In [76]:
alloc_cand[alloc_cand['allocated_against_ur'].notnull()].shape
# alloc_cand[alloc_cand['allocated_category'].notnull()].shape


(12941, 147)

In [77]:
updated_vacancy_df = pd.DataFrame.from_dict(upd_vacancy_dict, orient='index')

In [78]:
updated_vacancy_df[updated_vacancy_df['current'] != 0][['initial_vacancy', 'current', 'allocated', 'allocated_hc']].head()

,initial_vacancy,current,allocated,allocated_hc
21118-27,2,1,NaN,0
81118-27,2,1,NaN,0
21818-27,1,1,NaN,0
91818-27,5,5,NaN,0
22318-27,8,4,NaN,0


In [79]:
def update_vacancy_table(vacancy_table_df, category_code, state_code, age_limit, vacancy):
    condition = (
        (vacancy_table_df['category_code'].astype(str) == category_code) &
        (vacancy_table_df['state_code'].astype(str) == state_code) &
        (vacancy_table_df['age_limit'].astype(str) == age_limit)
    )
    if condition.any():
        current = vacancy_table_df.loc[condition, 'current']
        #print(vacancy_table_df[condition]['current'])
        vacancy_table_df.loc[condition, 'current'] -= vacancy
        #print(vacancy_table_df[condition]['current'])
        #print()
        
def update_horizontal_vacancy(vacancy_table_df, category_code, state_code, age_limit, vacancy):
    condition = (
        (vacancy_table_df['category_code'].astype(str) == category_code) &
        (vacancy_table_df['state_code'].astype(str) == state_code) &
        (vacancy_table_df['age_limit'].astype(str) == age_limit)
    )
    if condition.any():
        #print(vacancy_table_df[condition]['current'])
        vacancy_table_df.loc[condition, 'current'] = vacancy
        #print(vacancy_table_df[condition]['current'])
        #print()
        
def adjust_vacancy(candidates_df, vacancy_table_df):
    vacancy_table_df['current'] = vacancy_table_df['initial_vacancy']
    vacancy_table_df['allocated'] = 0
    vacancy_table_df['left_vacancy'] = 0

    grouped_candidates = candidates_df[candidates_df['allocated_category'].isin(['3', '4', '5', '7', '8'])].groupby(['cat1', 'allocated_state', 'age_limit', 'allocated_against_ur'])

    for (category_code, state_code, age_limit, allocated_against_ur), group in grouped_candidates:
        category_code = '9' if allocated_against_ur == '1' else category_code
        vacancy = len(group)
        
        update_vacancy_table(vacancy_table_df, category_code, state_code, age_limit, vacancy)

    '''grouped_candidates2 = candidates_df[candidates_df['allocated_category'].isin(['3', '4', '5', '7', '8'])].groupby(['allocated_category', 'allocated_state', 'age_limit', 'allocated_against_ur'])

    horizontal_vacancy = grouped_candidates2.size().reset_index(name='vacancy')
    #print(horizontal_vacancy)
    for index, row in horizontal_vacancy.iterrows():
        category_code, state_code, age_limit, vacancy = row['allocated_category'], row['allocated_state'], row['age_limit'], row['vacancy']

        #print('horizontal', category_code, state_code, age_limit,  vacancy)
        update_horizontal_vacancy(vacancy_table_df, category_code, state_code, age_limit, vacancy)'''

    vacancy_table_df['allocated_hc_prev'] = vacancy_table_df['allocated_hc']
    vacancy_table_df['allocated_hc'] = 0

    return vacancy_table_df


In [80]:
upd_vacancy_df = adjust_vacancy(alloc_cand, updated_vacancy_df)

In [81]:
upd_vacancy_df[upd_vacancy_df['current'] == 0][['initial_vacancy', 'current', 'allocated_hc_prev', 'allocated_hc']].head()

,initial_vacancy,current,allocated_hc_prev,allocated_hc
01818-27,1,0,1,0
67118-27,10,0,10,0
21218-25,0,0,0,0
31218-25,0,0,0,0
41218-25,0,0,0,0


In [67]:
#alloc_cand[alloc_cand['allocated_category'].notnull()].to_csv('allocated_data.csv', index = False)

In [68]:
#origin_candidate_df[origin_candidate_df['allocated_category'].notnull()].to_csv('allocated_data_origin.csv', index = False)

In [82]:
upd_vacancy_df[upd_vacancy_df['current'] != upd_vacancy_df['initial_vacancy']].shape

(163, 20)

In [83]:
upd_vacancy_df['left_vacancy'] = upd_vacancy_df['current']
vacancy_dict = {}
upd_vacancy_df['key'] = upd_vacancy_df['category_code'].astype(str) + upd_vacancy_df['state_code'].astype(str) + upd_vacancy_df['age_limit'].astype(str)

for index, row in upd_vacancy_df.iterrows():
    key = row['key']
    vacancy_dict[key] = row.to_dict()

In [84]:
alloc_cand[['age_limit', 'allocated_category', 'allocated_state', 'allocated_against_ur']] = None

##### 2nd Time

In [85]:
upd_vacancy_dict = allocate_candidates(alloc_cand, vacancy_dict, havaldar_state_lst)

In [86]:
alloc_cand[alloc_cand['allocated_against_ur'].notnull()].shape

(11507, 147)

In [134]:
# alloc_cand[alloc_cand['allocated_against_ur'].notnull()].to_csv('allocated_data_323.csv', index = False)

In [87]:
updated_vacancy_df = pd.DataFrame.from_dict(upd_vacancy_dict, orient='index')

In [88]:
upd_vacancy_df = adjust_vacancy(alloc_cand, updated_vacancy_df)

In [89]:
upd_vacancy_df['current'].sum()

11520

In [90]:
upd_vacancy_df['left_vacancy'] = upd_vacancy_df['current']
vacancy_dict = {}
upd_vacancy_df['key'] = upd_vacancy_df['category_code'].astype(str) + upd_vacancy_df['state_code'].astype(str) + upd_vacancy_df['age_limit'].astype(str)

for index, row in upd_vacancy_df.iterrows():
    key = row['key']
    vacancy_dict[key] = row.to_dict()

In [91]:
alloc_cand[['age_limit', 'allocated_category', 'allocated_state', 'allocated_against_ur']] = None

##### 3rd Time

In [92]:
upd_vacancy_dict = allocate_candidates(alloc_cand, vacancy_dict, havaldar_state_lst)

In [93]:
alloc_cand[alloc_cand['allocated_against_ur'].notnull()].shape

(11507, 147)

In [82]:
# alloc_cand[alloc_cand['allocated_against_ur'].notnull()].to_csv('allocated_data_323_1.csv', index = False)

In [94]:
updated_vacancy_df = pd.DataFrame.from_dict(upd_vacancy_dict, orient='index')

In [95]:
updated_vacancy_df

,post_name,post_code,state_code,state,region,age_limit,category,category_code,original_vacancy,initial_vacancy,current,allocated,left_vacancy,key,min_marks_prev,minpart4marks_prev,lowestmarkssession1_prev,min_marks_cand_dob_prev,allocated_hc,allocated_hc_prev
01118-27,HAVALDAR-CGST,HCG,11,Chandigarh- Hawaldar,NWR,18-27,EWS,0,11,11,0,11,11,01118-27,126.26082,35.0,55.58734,2006-06-12,0,0
11118-27,HAVALDAR-CGST,HCG,11,Chandigarh- Hawaldar,NWR,18-27,SC,1,10,10,0,10,10,11118-27,121.96142,31.0,66.46335,2005-09-18,0,0
21118-27,HAVALDAR-CGST,HCG,11,Chandigarh- Hawaldar,NWR,18-27,ST,2,2,2,1,1,2,21118-27,122.14574,35.0,74.60975,2006-05-01,0,0
31118-27,HAVALDAR-CGST,HCG,11,Chandigarh- Hawaldar,NWR,18-27,ESM,3,8,8,0,8,8,31118-27,90.18447,10.0,44.23924,1992-11-18,0,0
41118-27,HAVALDAR-CGST,HCG,11,Chandigarh- Hawaldar,NWR,18-27,OH,4,1,1,0,1,1,41118-27,122.72300,36.0,105.24359,2001-07-26,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56818-27,MTS,MTS,68,Kerala,KKR,18-27,HH,5,0,0,0,0,0,56818-27,NaN,NaN,NaN,NaN,0,0
66818-27,MTS,MTS,68,Kerala,KKR,18-27,OBC,6,7,7,0,6,6,66818-27,NaN,NaN,NaN,NaN,1,1
76818-27,MTS,MTS,68,Kerala,KKR,18-27,VH,7,1,1,0,1,1,76818-27,NaN,NaN,NaN,NaN,0,0
86818-27,MTS,MTS,68,Kerala,KKR,18-27,PWD-Others,8,2,2,0,2,2,86818-27,NaN,NaN,NaN,NaN,0,0


In [96]:
upd_vacancy_df = adjust_vacancy(alloc_cand, updated_vacancy_df)
updated_vacancy_df['current_prev'] = updated_vacancy_df['current']

In [97]:
upd_vacancy_df[upd_vacancy_df['current'] == 0][['initial_vacancy', 'current', 'allocated_hc_prev']]

,initial_vacancy,current,allocated_hc_prev
01818-27,1,0,1
67118-27,10,0,10
06018-27,1,0,1
21218-25,0,0,0
31218-25,0,0,0
...,...,...,...
56718-27,0,0,0
76718-27,0,0,0
86718-27,0,0,0
26818-27,0,0,0


In [98]:
upd_vacancy_df[upd_vacancy_df['current'] != upd_vacancy_df['initial_vacancy']].shape

(161, 21)

In [99]:
upd_vacancy_df['current'].sum()

11520

In [100]:
upd_vacancy_df['left_vacancy'] = upd_vacancy_df['current']
vacancy_dict = {}
upd_vacancy_df['key'] = upd_vacancy_df['category_code'].astype(str) + upd_vacancy_df['state_code'].astype(str) + upd_vacancy_df['age_limit'].astype(str)

for index, row in upd_vacancy_df.iterrows():
    key = row['key']
    vacancy_dict[key] = row.to_dict()

In [101]:
alloc_cand[['age_limit', 'allocated_category', 'allocated_state', 'allocated_against_ur']] = None

##### 4th Time

In [102]:
upd_vacancy_dict = allocate_candidates(alloc_cand, vacancy_dict, havaldar_state_lst)

In [103]:
alloc_cand[alloc_cand['allocated_against_ur'].notnull()].to_csv('allocated_data_final_revised.csv', index = False)

In [104]:
alloc_cand[alloc_cand['allocated_against_ur'].notnull()].shape

(11507, 147)

In [105]:
candidate_table_df['catsel_18_25'].unique()

array(['', '96', '91', '90', '1', '9', '0', '6', '2', '967', '92', '14',
       '964', '95', '97', '17', '7', '3', '13', '138', '98', '4', '917',
       '963', '5', '24', '64', '63', '68', '8', '968', '914', '94', '93',
       '913', '65', '67', '927', '18', '918', '924', '27', '965', '923',
       '904', '907', '04', '23', '905', '15', '903', '28', '908', '05',
       '634', '9634', '25', '928', '938', '38', '34', '915', '934', '935',
       '9034', '03', '925', '638', '35', '07', '9638', '08', '035',
       '9135', '134', '937', '37'], dtype=object)

In [ ]:
#alloc_cand[alloc_cand['allocated_against_ur'].notnull()].to_csv(r"C:\Users\Aviral Chaudhary\Downloads\allocated_data.csv", index = False)

In [106]:
updated_vacancy_df = pd.DataFrame.from_dict(upd_vacancy_dict, orient='index')

In [107]:
updated_vacancy_df['key'] = updated_vacancy_df['category_code'].astype(str) + updated_vacancy_df['state_code'].astype(str) + updated_vacancy_df['age_limit'].astype(str)

In [108]:
def find_lowest_marks(candidates_df, vacancy_df):
    try:
        vc = []
        vacancy_df['min_marks'] = 0
        for _, row in vacancy_df.iterrows():
            lmv = {
                "category_code": str(row["category_code"]),
                "state_code": str(row["state_code"]),
                "age_limit": str(row["age_limit"]),
            }

            vc.append(lmv)

        print("Vector size--->", len(vc))

        for lmv1 in vc:
            lowest_marks, lowest_part4, lowest_dob, lowest_session1 = get_lowest_marks(candidates_df, lmv1["category_code"], lmv1["state_code"], lmv1["age_limit"])
            update_vacancy_table(vacancy_df, lmv1["category_code"], lmv1["state_code"], lmv1["age_limit"], lowest_marks, lowest_part4, lowest_dob, lowest_session1)

    except Exception as e:
        print(e)

def get_lowest_marks(candidates_df, category_code, state_code, age_limit):
    min_marks = 0

    try:
        
        mask = (
            (candidates_df["allocated_category"] == str(category_code))
            & (candidates_df["allocated_state"] == str(state_code))
            & (candidates_df["age_limit"] == str(age_limit))
        )
        
        filtered_cand = candidates_df[mask]

        min_marks = filtered_cand['session2'].min()
        min_part4 = filtered_cand['part3_ga'].min()
        min_dob = filtered_cand['dob'].max()
        min_session1 = filtered_cand['session1'].min()

        #print(f"category_code {category_code} state_code {state_code} age_limit {age_limit} min_marks {min_marks}")

    except Exception as ex:
        print(ex)

    return min_marks, min_part4, min_dob, min_session1

def update_vacancy_table(vacancy_df, category_code, state_code, age_limit, min_marks, min_part4, min_dob, min_session1):
    try:
        key = str(category_code) + str(state_code) + str(age_limit)
        mask = (vacancy_df['key'] == key)
        vacancy = vacancy_df.loc[mask]
        vacancy["min_marks"] = min_marks
        vacancy["minpart4marks"] = min_part4
        vacancy["lowestmarkssession1"] = min_session1
        vacancy["min_marks_cand_dob"] = min_dob
        
        vacancy_df.loc[mask, 'min_marks'] = vacancy["min_marks"]
        vacancy_df.loc[mask, 'minpart4marks'] = vacancy["minpart4marks"]
        vacancy_df.loc[mask, 'lowestmarkssession1'] = vacancy["lowestmarkssession1"]
        vacancy_df.loc[mask, 'min_marks_cand_dob'] = vacancy["min_marks_cand_dob"]

        #print(f"category_code {category_code} state_code {state_code} min_marks {min_marks}")
        #print(f"Min Marks updated")

    except Exception as ex:
        print(ex)

find_lowest_marks(alloc_cand, updated_vacancy_df)


Vector size---> 783


C:\Users\AshutoshMishra\AppData\Local\Temp\ipykernel_12848\3742953887.py:58: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[128.44809]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  vacancy_df.loc[mask, 'min_marks'] = vacancy["min_marks"]


In [109]:
updated_vacancy_df.head(10)

,post_name,post_code,state_code,state,region,age_limit,category,category_code,original_vacancy,initial_vacancy,...,minpart4marks_prev,lowestmarkssession1_prev,min_marks_cand_dob_prev,allocated_hc,allocated_hc_prev,current_prev,min_marks,minpart4marks,lowestmarkssession1,min_marks_cand_dob
01118-27,HAVALDAR-CGST,HCG,11,Chandigarh- Hawaldar,NWR,18-27,EWS,0,11,11,...,35.0,55.58734,2006-06-12,0,0,11,128.44809,41.0,87.74733,2004-01-01
11118-27,HAVALDAR-CGST,HCG,11,Chandigarh- Hawaldar,NWR,18-27,SC,1,10,10,...,31.0,66.46335,2005-09-18,0,0,10,122.59747,32.0,73.59392,2005-09-18
21118-27,HAVALDAR-CGST,HCG,11,Chandigarh- Hawaldar,NWR,18-27,ST,2,2,2,...,35.0,74.60975,2006-05-01,0,0,2,128.44809,42.0,115.44202,1999-09-01
31118-27,HAVALDAR-CGST,HCG,11,Chandigarh- Hawaldar,NWR,18-27,ESM,3,8,8,...,10.0,44.23924,1992-11-18,0,0,8,94.02723,19.0,67.18645,1989-03-12
41118-27,HAVALDAR-CGST,HCG,11,Chandigarh- Hawaldar,NWR,18-27,OH,4,1,1,...,36.0,105.24359,2001-07-26,0,0,1,135.42337,60.0,94.36850,1990-02-18
51118-27,HAVALDAR-CGST,HCG,11,Chandigarh- Hawaldar,NWR,18-27,HH,5,1,1,...,28.0,91.19875,2004-04-17,0,0,1,126.57150,42.0,72.10703,1994-08-14
61118-27,HAVALDAR-CGST,HCG,11,Chandigarh- Hawaldar,NWR,18-27,OBC,6,28,28,...,27.0,62.29706,2006-02-28,4,4,24,129.32048,37.0,56.11722,2004-06-09
81118-27,HAVALDAR-CGST,HCG,11,Chandigarh- Hawaldar,NWR,18-27,OTH,8,2,2,...,31.0,83.25693,2005-03-10,0,0,2,120.54275,46.0,83.25693,1998-04-23
91118-27,HAVALDAR-CGST,HCG,11,Chandigarh- Hawaldar,NWR,18-27,UR,9,28,28,...,34.0,56.11722,2006-07-24,7,7,21,135.84341,47.0,79.86278,2006-07-24
01818-27,HAVALDAR-CGST,HCG,18,Delhi- Hawaldar,NR,18-27,EWS,0,1,1,...,43.0,85.08114,2004-03-23,1,1,0,NaN,NaN,NaN,NaT


In [110]:
print(alloc_cand.shape)  # Rows x Columns count
print(alloc_cand.head())  # First 5 rows


(2250749, 147)
                id        regno             otrId  myApplicationId  \
0  alo42etfewvb5gm  10015897919  4lgo612e7ka5j3nr  jcqm8rnvux17h1o   
1  42bs4vrw293bx8w  10015926809  bnt0tgnf7bcuz68v  sbxgh7lxlfkjmuc   
2  tewlvx7ih3nwqas  10015931276  x231rp3qhbh43hmi  u346ov02opb7fis   
3  3fli27xeehd5meo  10015932489  ubv3sy5218drbghv  sliqbz8fa7erjxh   
4  5izfi5ofgqn0onq  10016006653  mjmuy988rafqqal0  puul7axxy9l6mbk   

                     emailId     phoneNo             name changed_name  \
0   sandeep.kr6246@gmail.com  8877994789    SANDEEP KUMAR          NaN   
1  govindverma2118@gmail.com  9302982118    GOVIND PARMAR          NaN   
2       abhay42101@gmail.com  9334537186      ABHAY KUMAR          NaN   
3         jugalc54@gmail.com  9589674139  JUGAL CHOUDHARY          NaN   
4   computervikash@gmail.com  8709301761     CHAND ANSARI          NaN   

       father_name             mother_name  ... dob_flag_18_25  \
0     SANJAY KUMAR  KUMARI ANITA CHOUDHARY  ...      

In [111]:
updated_vacancy_df.to_csv("allocated_vacancy_final_revised.csv", index = False)

In [112]:
alloc_cand.to_csv('complete_data_final_mts_revised.csv', index = False)

In [ ]:
import pandas as pd

# Load the large CSV file
csv_file = r"C:\Users\AshutoshMishra\OneDrive - Cubastion Consulting Pvt Ltd\Desktop\SSC OPS\RESULT PROCESSING MTS 2024\complete_data_final_mts.csv"
df = pd.read_csv(csv_file)

# Total rows
total_rows = len(df)
print(f"Total Rows: {total_rows}")

# Number of files
num_files = 5
rows_per_file = total_rows // num_files  # Divide into 5 equal parts

# Split and save files
for i in range(num_files):
    start_row = i * rows_per_file
    end_row = (start_row + rows_per_file) if i < num_files - 1 else total_rows  # Last file takes remaining rows
    file_name = f"split_data_part_{i+1}.csv"
    
    df.iloc[start_row:end_row].to_csv(file_name, index=False)
    print(f"{file_name} saved with rows {start_row} to {end_row}")

print("✅ All 5 files saved successfully!")


In [ ]:
alloc_cand[alloc_cand['merit']==1128]['state_ut_pref'].to_list()

In [ ]:
alloc_cand[alloc_cand['merit']==1128][['catsel_18_27', 'catsel_18_25']]

In [ ]:
alloc_cand[alloc_cand['merit']==1128]

In [ ]:
origin_candidates_df = pd.read_csv(r"C:\Users\Aviral Chaudhary\Downloads\candidates\candidates.csv")

In [ ]:
origin_candidates_df = origin_candidates_df.sort_values(by = 'merit')

In [ ]:
origin_candidates_df = origin_candidates_df.head(200000)

In [ ]:
origin_candidates_df.to_csv(r"C:\Users\Aviral Chaudhary\Downloads\candidates\candidates_top_200000.csv", index = False)

In [ ]:
(alloc_cand.sort_values(by = 'merit').head(200000)).to_csv(r"alloc_candidates_top_200000.csv", index = False)